# Retail Orders — Data Profiling & Quality Contract
**Purpose:** Profile the supplied raw retail order data, run executable data-quality checks, and produce evidence for the KPI dictionary.

## How to run
1. Keep `retail-orders-raw.csv` in the same folder as this notebook (or update `ORDERS_PATH`).
2. Run all cells from top to bottom.
3. Review `quality_results.csv`, `column_profile.csv`, and the printed exception rows.

**Important assumptions:** Dates are parsed day-first only as a documented working assumption; confirm `03/01/2026` with the source owner. Missing discounts are not automatically replaced with zero. Duplicate IDs are flagged, not silently removed.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import date

ORDERS_PATH = Path("retail-orders-raw.csv")
if not ORDERS_PATH.exists():
    # Convenient fallback for this task's mounted environment
    fallback = Path("/mnt/data/retail-orders-raw.csv")
    if fallback.exists():
        ORDERS_PATH = fallback

raw = pd.read_csv(ORDERS_PATH, dtype={"order_id": "string"})
print(f"Loaded {len(raw)} rows and {len(raw.columns)} columns from {ORDERS_PATH}")
display(raw.head())


## 1. Basic profiling

In [ ]:
profile = pd.DataFrame({
    "column": raw.columns,
    "dtype": [str(raw[c].dtype) for c in raw.columns],
    "missing_count": [raw[c].isna().sum() + raw[c].astype("string").str.strip().eq("").sum() for c in raw.columns],
    "missing_pct": [round((raw[c].isna().sum() + raw[c].astype("string").str.strip().eq("").sum()) / len(raw) * 100, 2) for c in raw.columns],
    "distinct_non_null": [raw[c].nunique(dropna=True) for c in raw.columns],
})
display(profile)
profile.to_csv("column_profile.csv", index=False)


## 2. Normalize and validate fields

In [ ]:
df = raw.copy()
# Normalize text fields without altering the raw source.
text_cols = ["order_id", "customer_segment", "city", "category", "payment_status"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()
df["customer_segment_norm"] = df["customer_segment"].str.lower().map({
    "student": "Student", "fresher": "Fresher", "professional": "Professional"
})
df["category_norm"] = df["category"].map({
    "Learning Kit": "Learning Kit", "Course Access": "Course Access", "Mentor Session": "Mentor Session"
})
df["payment_status_norm"] = df["payment_status"].str.lower().map({
    "paid": "Paid", "pending": "Pending", "failed": "Failed", "refunded": "Refunded"
})
# dayfirst=True is a documented assumption for ambiguous values such as 03/01/2026.
df["order_date_parsed"] = pd.to_datetime(df["order_date"], errors="coerce", dayfirst=True)
df["quantity_num"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price_num"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["discount_pct_num"] = pd.to_numeric(df["discount_pct"], errors="coerce")
today = pd.Timestamp.today().normalize()
allowed_categories = {"Learning Kit", "Course Access", "Mentor Session"}
allowed_segments = {"Student", "Fresher", "Professional"}
allowed_statuses = {"Paid", "Pending", "Failed", "Refunded"}
display(df[["order_id", "order_date", "order_date_parsed", "customer_segment", "customer_segment_norm",
            "city", "category", "quantity", "quantity_num", "unit_price_num", "discount_pct_num",
            "payment_status", "payment_status_norm"]])


## 3. Execute data-quality contract

In [ ]:
checks = []
def add_check(rule_id, field, dimension, failures, denominator, threshold, severity, detail):
    failures = int(failures)
    denominator = int(denominator)
    pass_rate = (denominator - failures) / denominator * 100 if denominator else np.nan
    checks.append({
        "rule_id": rule_id, "field": field, "dimension": dimension,
        "failed_records": failures, "records_tested": denominator,
        "pass_rate_pct": round(pass_rate, 2) if pd.notna(pass_rate) else np.nan,
        "threshold": threshold, "severity": severity,
        "status": "PASS" if failures == 0 else "FAIL", "detail": detail
    })

n = len(df)
add_check("DQ-01","order_id","Completeness",df["order_id"].isna().sum() + df["order_id"].eq("").sum(),n,"100% required","Critical","Missing/blank order IDs")
add_check("DQ-02","order_id","Uniqueness",df["order_id"].duplicated(keep=False).sum(),n,"No duplicate IDs","Critical","Counts rows participating in duplicate IDs")
date_bad = df["order_date_parsed"].isna() | (df["order_date_parsed"] < pd.Timestamp("2025-01-01")) | (df["order_date_parsed"] > today)
add_check("DQ-03","order_date","Validity",date_bad.sum(),n,"100% valid dates in range","High","Unparseable, missing, too early, or future dates")
seg_bad = ~df["customer_segment_norm"].isin(allowed_segments)
add_check("DQ-04","customer_segment","Domain",seg_bad.sum(),n,"Allowed normalized values only","High","Unknown or missing customer segment")
city_bad = df["city"].isna() | df["city"].eq("")
add_check("DQ-05","city","Completeness",city_bad.sum(),n,">=99% (target 100%)","Medium","Missing/blank billing city")
cat_bad = ~df["category_norm"].isin(allowed_categories)
add_check("DQ-06","category","Domain",cat_bad.sum(),n,"Allowed categories only","High","Unknown or missing category")
qty_bad = df["quantity_num"].isna() | (df["quantity_num"] <= 0) | (df["quantity_num"] % 1 != 0)
add_check("DQ-07","quantity","Type/range",qty_bad.sum(),n,"Whole number > 0","Critical","Non-numeric, fractional, zero, or negative quantity")
price_bad = df["unit_price_num"].isna() | (df["unit_price_num"] < 0)
add_check("DQ-08","unit_price","Range",price_bad.sum(),n,"Numeric INR amount >= 0","High","Invalid or negative unit price")
discount_bad = df["discount_pct_num"].isna() | (df["discount_pct_num"] < 0) | (df["discount_pct_num"] > 100)
add_check("DQ-09","discount_pct","Range/completeness",discount_bad.sum(),n,"0–100; missing needs approved policy","High","Missing or out-of-range discount")
status_bad = ~df["payment_status_norm"].isin(allowed_statuses)
add_check("DQ-10","payment_status","Domain",status_bad.sum(),n,"Allowed normalized statuses only","High","Unknown or missing payment status")
quality_results = pd.DataFrame(checks)
display(quality_results)
quality_results.to_csv("quality_results.csv", index=False)


## 4. Exception rows and duplicate review

In [ ]:
exception_mask = (
    df["order_id"].duplicated(keep=False) | date_bad | seg_bad | city_bad | cat_bad |
    qty_bad | price_bad | discount_bad | status_bad
)
exception_cols = ["order_id","order_date","customer_segment","city","category","quantity","unit_price","discount_pct","payment_status"]
exceptions = df.loc[exception_mask, exception_cols].copy()
display(exceptions)
exceptions.to_csv("exception_rows.csv", index=False)

duplicate_review = df[df["order_id"].duplicated(keep=False)].sort_values("order_id")
display(duplicate_review[exception_cols])


## 5. KPI calculations on eligible records (illustrative)

In [ ]:
# Conservative eligibility: unique order IDs, valid date, positive whole quantity,
# non-negative price, valid discount, recognized payment status.
# Missing discounts remain excluded pending owner decision.
eligible = df[
    ~df["order_id"].isna() & ~df["order_id"].duplicated(keep=False) &
    ~date_bad & ~qty_bad & ~price_bad & ~discount_bad &
    df["payment_status_norm"].isin(allowed_statuses)
].copy()
eligible["gross_value"] = eligible["quantity_num"] * eligible["unit_price_num"]
eligible["net_value"] = eligible["gross_value"] * (1 - eligible["discount_pct_num"] / 100)
sales_scope = eligible[~eligible["payment_status_norm"].isin(["Failed", "Refunded"])]
kpi_summary = pd.DataFrame([
    ["Raw rows", len(raw), "All source rows"],
    ["Distinct non-null order IDs", df["order_id"].nunique(dropna=True), "Before deduplication"],
    ["Eligible unique rows for illustrative KPIs", len(eligible), "Conservative validation filter"],
    ["Net sales (INR)", round(sales_scope["net_value"].sum(), 2), "Excludes Failed/Refunded; Pending retained separately by policy decision"],
    ["Units sold", int(sales_scope["quantity_num"].sum()), "Same conservative sales scope"],
    ["Average order value (INR)", round(sales_scope["net_value"].mean(), 2) if len(sales_scope) else np.nan, "Average per eligible order"],
    ["Paid orders", int((eligible["payment_status_norm"] == "Paid").sum()), "Eligible distinct orders"],
], columns=["KPI","Value","Scope / caveat"])
display(kpi_summary)
kpi_summary.to_csv("kpi_summary.csv", index=False)


## 6. Escalation and release decision
- **Critical failures** (missing/duplicate order IDs, invalid quantity): block publication of affected KPI outputs until reviewed.
- **High failures** (invalid date, category, price, discount, payment status): quarantine affected records and notify the Data Steward / domain owner.
- **Medium failures** (missing city, freshness): publish only if the affected analysis is clearly caveated and owner-approved.
- Never silently discard duplicates or convert missing discounts to zero.
- Agree the freshness SLA (proposed 09:00 local time daily) with the data owner before production use.

## Deliverables produced by this notebook
`column_profile.csv`, `quality_results.csv`, `exception_rows.csv`, and `kpi_summary.csv`.
